# Schottky, MSM and phototransistors {#sec-schottky}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/schottky.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/schottky.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter5/detlib.py', 'detlib.py')
else:
    sys.path.insert(0, '..')                 # detlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import detlib as dl
dl.use_style()

*Slides 61–65 (Schottky junction, barrier, dark current, Schottky photodiodes and their characteristics), 66–67
(metal–semiconductor–metal detectors with interdigitated electrodes) and 68 (phototransistor).*

## Key relations

**Schottky photodiode.** A metal with a work function $\Phi_m$ larger than that of an n-type semiconductor (electron affinity
$\chi$) forms a rectifying contact: electrons flow from the semiconductor into the metal until the Fermi levels align, leaving a
depletion region in the semiconductor and a barrier (slide 62)

$$
\Phi_B=\Phi_m-\chi
$$

for electrons in the metal. In reverse bias the dark current is the thermionic emission of electrons from the metal over this
barrier (slide 63):

$$
J_\text{dark}\approx A^*T^2\exp\!\left(-\frac{\Phi_B}{k_BT}\right),\qquad A^*=\frac{4\pi em^*k_B^2}{h^3}\approx1.2\times10^6\frac{m^*}{m_e}\ \text{A m}^{-2}\text{K}^{-2}.
$$

Light with $h\nu>E_g$ passes through a thin, semitransparent metal and is absorbed in the depletion region, right at the surface
(slide 64). There is no neutral p⁺ "dead layer" as in a pn diode, so Schottky photodiodes have an excellent **UV and blue
response** and are **fast**: only drift in the field, no minority-carrier diffusion (table of slide 65: GaAsP, GaP, AlGaN for UV;
GaAs with $\tau_r<30$ ps).

**MSM detector.** Two Schottky contacts back to back on the same surface, usually as **interdigitated fingers** (slides 66–67). Under
bias the region between the fingers is fully depleted. Both the transit length (the finger gap $s$) and the capacitance (planar
electrodes, no parallel-plate junction) are small, so MSM detectors reach tens of GHz. The price: the fingers shadow part of the
area (fill factor $s/(s+w_f)$), and the field is weak deep below the surface.

**Phototransistor.** A bipolar transistor whose base–collector junction is the photodiode (slide 68). The photogenerated holes
swept into the base act as base current, and transistor action multiplies the collector current:

$$
I_{ph}\approx\beta I_{pho}.
$$

The gain is free, but the base must be charged through the emitter resistance $r_e=k_BT/eI_C$: the bandwidth
$f_{3dB}\approx1/(2\pi\beta r_eC)=I_{pho}/(2\pi(k_BT/e)C)$ is proportional to the primary photocurrent. Bright light makes it fast,
dim light makes it slow.

## Live panel: Schottky dark current and internal photoemission

A Schottky diode also responds to photons with $E_g>h\nu>\Phi_B$: the photon is absorbed by the electrons of the **metal**, and
the hot electrons that cross the barrier are collected (**internal photoemission**). Fowler's law gives the yield
$\eta=C_1(h\nu-\Phi_B)^2/h\nu$ with $C_1\sim0.1$–$0.5$ eV⁻¹. It is weak (≈ 1 %), but it lets a silicon chip see the mid-infrared:
PtSi/Si ($\Phi_B\approx0.22$ eV) infrared cameras work in the 3–5 µm window.

```{ojs}
//| echo: false
import {kT} from "./det_sim.js"
viewof sk = Inputs.form({
  phi: Inputs.range([0.1, 1.4], {value: 0.9, step: 0.01, label: "barrier height Φ_B (eV)"}),
  Astar: Inputs.select(new Map([["GaAs (8.2×10⁴ A m⁻² K⁻²)", 8.16e4], ["Si (1.12×10⁶)", 1.12e6], ["GaP (1.0×10⁶)", 1.0e6]]), {label: "semiconductor A*"}),
  T: Inputs.range([60, 400], {value: 300, step: 1, label: "temperature T (K)"}),
  C1: Inputs.range([0.05, 1], {value: 0.2, step: 0.01, label: "Fowler coefficient C₁ (eV⁻¹)"}),
  Eg: Inputs.range([0.5, 3.5], {value: 1.12, step: 0.01, label: "band gap of the semiconductor E_g (eV)"})
})
```

```{ojs}
//| echo: false
skOut = {
  const J = sk.Astar * sk.T ** 2 * Math.exp(-sk.phi / kT(sk.T)), pts = [];
  for (let i = 0; i <= 500; i++) {
    const wl = 0.2 + 1.2 * 1.23984 / sk.phi * i / 500, E = 1.23984 / wl;
    const eta = E >= sk.Eg ? 0.8 : (E > sk.phi ? sk.C1 * (E - sk.phi) ** 2 / E : 0);
    pts.push({wl, R: eta * wl / 1.23984, mech: E >= sk.Eg ? "band-to-band (η ≈ 0.8)" : "internal photoemission"});
  }
  return {J, pts};
}
Plot.plot({
  width: Math.min(width, 1050), height: 280, x: {label: "wavelength (µm)"}, y: {label: "responsivity (A/W)", type: "log", domain: [1e-5, 2], grid: true},
  color: {legend: true, domain: ["band-to-band (η ≈ 0.8)", "internal photoemission"], range: ["#2a78d6", "#e34948"]},
  marks: [Plot.dot(skOut.pts.filter(d => d.R > 0), {x: "wl", y: "R", fill: "mech", r: 1.5})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">dark current density J = A*T²exp(−Φ<sub>B</sub>/kT) = <b>${skOut.J.toExponential(2)} A/m²</b> =
${(skOut.J * 1e-6 * 1e12).toPrecision(3)} pA/mm² · internal-photoemission cut-off λ = 1.24/Φ<sub>B</sub> = <b>${(1.23984 / sk.phi).toFixed(2)} µm</b> ·
band-gap cut-off ${(1.23984 / sk.Eg).toFixed(2)} µm</div>`
```

The GaAsP and GaP photodiodes of slide 65 have dark currents of 5 pA per mm²: with the GaAs Richardson constant this needs
$\Phi_B\approx0.9$ eV. Lower the barrier to 0.22 eV (PtSi on Si, $E_g=1.12$ eV) and set $T=77$ K: the cut-off moves to 5.6 µm, the
QE is about 1 %, and the dark current is only acceptable because the camera is cooled. Every 60 meV of barrier ($\times\ln10$ of
$k_BT$) is a decade of dark current at 300 K.

## Live panel: MSM finger geometry

```{ojs}
//| echo: false
ellK = (k) => { let a = 1, b = Math.sqrt(1 - k * k); for (let i = 0; i < 30; i++) { const an = (a + b) / 2; b = Math.sqrt(a * b); a = an; } return Math.PI / (2 * a); }
viewof msm = Inputs.form({
  side: Inputs.range([10, 200], {value: 50, step: 1, label: "detector side (µm, square)"}),
  wf: Inputs.range([0.1, 5], {value: 1, step: 0.05, label: "finger width w_f (µm)"}),
  s: Inputs.range([0.1, 10], {value: 1, step: 0.05, label: "finger gap s (µm)"}),
  er: Inputs.range([9, 13], {value: 12.9, step: 0.1, label: "ε_r of the substrate (GaAs 12.9)"}),
  v: Inputs.range([2e4, 2e5], {value: 7e4, step: 1e3, label: "slower saturation velocity (m/s)"})
})
```

```{ojs}
//| echo: false
msmOut = {
  const eps0 = 8.8541878128e-12, A = (msm.side * 1e-6) ** 2;
  const cap = (wf, s) => { const k = Math.tan(Math.PI * wf / (4 * (wf + s))) ** 2; return eps0 * (1 + msm.er) * ellK(k) / ellK(Math.sqrt(1 - k * k)) * A / ((wf + s) * 1e-6); };
  const calc = (s) => { const C = cap(msm.wf, s), fRC = 1 / (2 * Math.PI * 50 * C), ftr = 0.45 * msm.v / (s * 1e-6);
    return {s, C, fRC, ftr, f: 1 / Math.sqrt(1 / fRC ** 2 + 1 / ftr ** 2), ff: s / (s + msm.wf)}; };
  return {cur: calc(msm.s), sweep: d3.range(0, 81).map(i => calc(0.1 * 100 ** (i / 80)))};
}
Plot.plot({
  width: Math.min(width, 1050), height: 280, x: {label: "finger gap s (µm)", type: "log"}, y: {label: "bandwidth (GHz)", type: "log", grid: true},
  color: {legend: true, domain: ["total (50 Ω)", "transit 0.45v/s", "RC"], range: ["#0b0b0b", "#2a78d6", "#e34948"]},
  marks: [Plot.line(msmOut.sweep, {x: "s", y: d => d.f / 1e9, stroke: () => "total (50 Ω)", strokeWidth: 2.5}),
          Plot.line(msmOut.sweep, {x: "s", y: d => d.ftr / 1e9, stroke: () => "transit 0.45v/s", strokeDasharray: "5,3"}),
          Plot.line(msmOut.sweep, {x: "s", y: d => d.fRC / 1e9, stroke: () => "RC", strokeDasharray: "5,3"}),
          Plot.dot([msmOut.cur], {x: "s", y: d => d.f / 1e9, r: 6, fill: "#1baf7a"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">capacitance <b>${(msmOut.cur.C * 1e15).toFixed(1)} fF</b> · f<sub>RC</sub> (50 Ω) = ${(msmOut.cur.fRC / 1e9).toFixed(1)} GHz ·
f<sub>transit</sub> ≈ ${(msmOut.cur.ftr / 1e9).toFixed(1)} GHz · <b>f<sub>3dB</sub> ≈ ${(msmOut.cur.f / 1e9).toFixed(1)} GHz</b> ·
optical fill factor s/(s + w<sub>f</sub>) = <b>${(100 * msmOut.cur.ff).toFixed(0)} %</b></div>`
```

Narrow gaps are fast but the fingers then cover a large part of the surface; with semitransparent ITO fingers or back
illumination the fill factor recovers. The capacitance of a 50 µm × 50 µm MSM is below 0.1 pF, several times less than a pin
of the same area with a 1 µm depletion layer.

## The phototransistor: gain for bandwidth

In [ ]:
#| label: fig-photoT
#| fig-cap: "Phototransistor bandwidth f ≈ I_pho/(2π(kT/e)C) versus primary photocurrent for C = 10 pF (base–collector plus base–emitter capacitance), compared with a pin photodiode on a 10 kΩ load (same capacitance). The phototransistor gives β = 100 times more current but is orders of magnitude slower at low light levels."
Ipho = np.logspace(-11, -5, 200)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.loglog(Ipho, dl.phototransistor_bandwidth(Ipho, 100, 10e-12), label="phototransistor (β = 100, C = 10 pF)")
ax.loglog(Ipho, np.full_like(Ipho, dl.rc_cutoff(1e4, 10e-12)), "--", label="pin, 10 kΩ, 10 pF")
ax.set(xlabel="primary photocurrent I_pho (A)", ylabel="f₃dB (Hz)"); ax.legend(fontsize=8); plt.show()

## The Schottky detectors of slide 65

In [ ]:
#| label: tbl-65
#| tbl-cap: "Schottky photodiodes of slide 65: QE implied by the peak responsivity, and the RC-equivalent bandwidth 0.35/τ_r implied by the rise time."
data = [("GaAsP", 610, 0.18, 3.5e-6), ("GaP", 440, 0.12, 5e-6), ("AlGaN", 350, 0.13, None), ("GaAs", 830, 0.2, 30e-12),
        ("InGaAs MSM", 1300, 0.4, 80e-12), ("GaAs MSM", 850, 0.3, 30e-12)]
pd.DataFrame([(n, l, R, f"{dl.quantum_efficiency(R, l*1e-9):.0%}", "–" if t is None else (f"{0.35/t/1e9:.1f} GHz" if t < 1e-9 else f"{0.35/t/1e3:.0f} kHz"))
              for n, l, R, t in data], columns=["detector", "λ_peak (nm)", "R (A/W)", "QE", "0.35/τ_r"])

The large-area GaAsP and GaP photometry diodes are slow (µs) because of their capacitance and high-impedance loads, not because of
the Schottky physics; the small GaAs and MSM devices reach 4–12 GHz. The QE of the MSM detectors (38–44 %) shows the finger shadowing.

::: {.callout-tip collapse="true"}
## Going further
- **Image-force lowering.** The barrier is lowered by $\Delta\Phi=\sqrt{eE/4\pi\varepsilon}$ at the field $E$ of the junction, so the
  "saturated" dark current of a Schottky diode increases with reverse bias. Compute $\Delta\Phi$ at $10^7$ V/m in GaAs.
- **Weighting field of an MSM** (@sec-ramo). Near the fingers the weighting field is large; carriers generated deep below the surface
  drift slowly in a weak field and produce a slow tail. Thin absorbing layers on an insulating or heterojunction barrier remove it.
- **Solar-blind UV detection.** AlGaN with $E_g>4.4$ eV (λ < 280 nm) sees the UV-C of flames and corona discharges without being
  blinded by sunlight, which the ozone layer has stripped of λ < 290 nm. Which Al fraction is needed?
- **Photogating and 2D materials.** Graphene and MoS₂ "photodetectors" with responsivities of $10^3$–$10^6$ A/W are phototransistors or
  photoconductors with long trap lifetimes (@sec-photoconductors). Apply the gain–bandwidth argument to judge such claims.
:::

::: {.callout-note collapse="true"}
## Complements
- Kasap, sec. 5.8–5.9; Sze & Ng, ch. 3 (metal–semiconductor contacts) and sec. 13.4.
- J. B. D. Soole and H. Schumacher, "InGaAs metal–semiconductor–metal photodetectors for long wavelength optical communications",
  *IEEE J. Quantum Electron.* 27, 737 (1991).
- Y. C. Lim and R. A. Moore, "Properties of alternately charged coplanar parallel strips by conformal mappings", *IEEE Trans.
  Electron Devices* 15, 173 (1968): the MSM capacitance formula.
:::

## Try it yourself

1. What barrier height gives a dark current of 1 pA/mm² in AlGaN ($A^*\approx3\times10^5$ A m⁻² K⁻²) at 300 K and at 350 K?
2. An InGaAs MSM with 1 µm fingers and 1 µm gaps covers 50 µm × 50 µm. Estimate its capacitance and bandwidth on 50 Ω. Why is
   its QE in slide 65 only 38 %?
3. A phototransistor with β = 500 and $C=5$ pF detects 1 nW at 850 nm ($\mathcal R=0.5$ A/W). What output current and what bandwidth
   do you expect? Would a pin with a 1 MΩ transimpedance amplifier do better?